# Host–Guest Committor: Descriptor Importance Analysis

Combines gradients, ordinary and conditional permutation tests, and HIPR to assess which descriptors control the learned committor.

This notebook is independently runnable: it imports its dependencies, resolves
the project paths, loads the RPE trainset and trained committor model, creates
the `SystemVisualizer`, and restores or computes the model-output cache.


## 1. Imports and shared analysis helpers


In [ ]:
import os
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import aimmd
import aimmd.pytorch
import aimmd.pytorch.rcmodel
import aimmdTIS
from aimmdTIS import Tools as TAI_Tools
from aimmdTIS.visualization import SystemVisualizer



## 2. Project and model configuration


In [ ]:
DEFAULT_WORK_DIR = Path.cwd()
WORK_DIR = Path(os.environ.get("HG_COMMITTOR_WORK_DIR", DEFAULT_WORK_DIR)).expanduser().resolve()
ANALYSIS_DIR = WORK_DIR

if not ANALYSIS_DIR.is_dir():
    raise FileNotFoundError(
        f"Analysis directory not found: {ANALYSIS_DIR}. "
        "Set HG_COMMITTOR_WORK_DIR to the project root."
    )
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

from committor_analysis_helpers import (
    aimmd_softplus_loss_terms,
    conditional_permutation_importance,
    descriptor_names,
    descriptor_swap_loss_matrix,
    permutation_influence_correlation,
    plot_correlated_pair,
    plot_square_matrix,
    predict_q,
    run_descriptor_correlation_tools,
    sigmoid_stable,
    subsample_arrays,
    weighted_gradient_importance,
    weighted_permutation_importance,
)

TRAINSET_FILE = WORK_DIR / "Files" / "trainset_rpe.pkl"
MODEL_DIR = WORK_DIR / "Files"
MODEL_FILE = MODEL_DIR / "aimmd_store_s5em04.h5"
MODEL_KEY = "stage1_full_model_model_best"
OUTPUT_DIR = WORK_DIR / "Figures"
CACHE_DIR = MODEL_DIR / ".cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

for required_file in (TRAINSET_FILE, MODEL_FILE):
    if not required_file.is_file():
        raise FileNotFoundError(required_file)

DESCRIPTOR_LABELS = [
    "Host-Guest\n Distance",
    "Guest Orientation\n Angle",
    "Guest Position\n Angle",
    "Hydrogen Bonds",
    "Waters in Cavity",
    "Hydrophobic\n Contact Score",
    "Shared Waters",
]
DESCRIPTOR_UNITS = ["nm", "rad", "rad", "", "", "", ""]
DESC_MIN = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
DESC_MAX = np.array([1.0, np.pi, np.pi, 4.0, 10.0, 60.0, 10.0])
N_BINS_DIM = np.array([200, 200, 200, 6, 50, 200, 50])
STATE_A, STATE_B = 0.05, 1.0
Q_MIN, Q_MAX = -45.0, 25.0
N_DESC = len(DESCRIPTOR_LABELS)


## 3. Load the RPE trainset and committor model


In [ ]:
clear_unbound = True

with TRAINSET_FILE.open("rb") as handle:
    trainset = pickle.load(handle)

required_keys = {"descriptors", "weights", "shot_results"}
missing_keys = required_keys.difference(trainset)
if missing_keys:
    raise KeyError(f"Trainset is missing keys: {sorted(missing_keys)}")

mask_clear_unbound = trainset["descriptors"][:, 0] <= 1.1 if clear_unbound else np.ones(len(trainset["descriptors"]), dtype=bool)

descriptors = np.asarray(trainset["descriptors"][mask_clear_unbound], dtype=float)
weights = np.asarray(trainset["weights"][mask_clear_unbound], dtype=float).reshape(-1)
shot_results = np.asarray(trainset["shot_results"][mask_clear_unbound])

if descriptors.ndim != 2 or descriptors.shape[1] != N_DESC:
    raise ValueError(f"Expected descriptors with shape (N, {N_DESC}), got {descriptors.shape}.")
if len(weights) != len(descriptors) or shot_results.shape != (len(descriptors), 2):
    raise ValueError("Trainset descriptors, weights, and shot results are not aligned.")
if np.any(weights < 0) or not np.all(np.isfinite(weights)):
    raise ValueError("Trainset weights must be finite and non-negative.")

positive_weights = weights[weights > 0]
if positive_weights.size == 0:
    raise ValueError("Trainset contains no positive weights.")
weights = weights / positive_weights.min()
trainset["weights"] = weights
trainset["descriptors"] = descriptors
trainset["shot_results"] = shot_results

storage = aimmd.Storage(str(MODEL_FILE), mode="r")
if MODEL_KEY not in storage.rcmodels:
    raise KeyError(f"Model key {MODEL_KEY!r} not found; available: {list(storage.rcmodels)}")
model = storage.rcmodels[MODEL_KEY]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TAI_Tools.model_to(model, device)

print(f"Frames: {len(descriptors):,}; descriptors: {descriptors.shape[1]}")
print(f"Shot results: {shot_results.shape[0]:,} frames, {shot_results.shape[1]} shots each")
print(f"weights: {weights.shape[0]:,} frames, min={weights.min():.3g}, max={weights.max():.3g}")
print(f"Weight range: {weights.min():.3g} – {weights.max():.3g}")
print(f"Model: {MODEL_KEY} on {device}")


## 4. Create and initialize the `SystemVisualizer`


In [ ]:
standard_value = np.average(descriptors, axis=0).tolist()
vis = SystemVisualizer(
    temperature=1.0,
    dims_extent=[0.0, 1.2, 0.0, float(np.pi)],
    total_num_descriptors=N_DESC,
    standard_value=standard_value,
    descriptor_labels=DESCRIPTOR_LABELS,
    desc_min=DESC_MIN,
    desc_max=DESC_MAX,
)
vis.load_trainset(trainset)
vis.plot_settings.fontsize = 15

print(f"Visualizer descriptors: {vis._desc.shape}")
print(f"Visualizer physical descriptors: {vis._desc_phys.shape}")


## 5. Restore or compute model outputs


In [ ]:
# Separate cache when unbound frames are removed (clear_unbound), since the frame set differs
Q_CACHE_FILE = CACHE_DIR / ("q_model_mbar_clear_unbound.npy" if clear_unbound else "q_model_mbar.npy")
RECOMPUTE_Q_MODEL = False  # set to True after changing the model or trainset
if Q_CACHE_FILE.is_file() and not RECOMPUTE_Q_MODEL:
    q_model = np.load(Q_CACHE_FILE)
    if q_model.shape != (len(descriptors),):
        raise ValueError(
            f"Cached model output has shape {q_model.shape}; expected {(len(descriptors),)}. "
            "Delete the stale cache and rerun this cell."
        )
    p_B = sigmoid_stable(q_model)
    vis._model_output_cache[model] = (p_B, q_model)
    print(f"Loaded model output from {Q_CACHE_FILE}")
else:
    p_B, q_model = vis._model_output_rpe(model)
    np.save(Q_CACHE_FILE, q_model)
    print(f"Computed and cached model output at {Q_CACHE_FILE}")

print(f"q range: {q_model.min():.3f} – {q_model.max():.3f}")


## 6. Prepare a reproducible analysis sample


In [ ]:
valid = np.sum(vis._shot, axis=1) > 0
X_all = np.asarray(vis._desc)[valid]
shots_all = np.asarray(vis._shot)[valid]
weights_all = np.asarray(vis._w)[valid]

X_sample, w_sample, shots_sample, sample_indices = subsample_arrays(
    X_all,
    weights=weights_all,
    shots=shots_all,
    max_samples=250_000,
    random_state=2026,
)
names = descriptor_names(N_DESC, DESCRIPTOR_LABELS)
print(f"Importance sample: {len(X_sample):,} frames")


## 7. Descriptor gradients along the committor


In [ ]:
Q_BINS_GRAD = np.unique(np.concatenate([
    np.arange(Q_MIN, -20.0, 0.1),
    np.arange(-20.0, -5.0, 0.1),
    np.arange(-5.0, 5.0, 0.1),
    np.arange(5.0, Q_MAX + 0.5, 0.1),
]))
NORM_FACTORS = np.max(vis._desc, axis=0) - np.min(vis._desc, axis=0)
NORM_FACTORS[0] = 1.0  
GRADIENT_CACHE_FILE = CACHE_DIR / "gradient_stats_variable_bins_v2.npz"
Recompute = False  # Set to True to force recomputation of gradient statistics
if GRADIENT_CACHE_FILE.is_file() and not Recompute:
    gradient_stats = vis.load_gradient_cache(model, GRADIENT_CACHE_FILE)
    print(f"Loaded gradient statistics from {GRADIENT_CACHE_FILE}")
else:
    gradient_stats = vis.compute_gradient_stats_per_q(
        model,
        norm_factors=NORM_FACTORS,
        q_bins=Q_BINS_GRAD,
        use_cache=False,
    )
    vis.save_gradient_cache(model, GRADIENT_CACHE_FILE)
    print(f"Computed and cached gradient statistics at {GRADIENT_CACHE_FILE}")

fig, ax = plt.subplots(figsize=(12, 4))
vis.plot_gradient_along_q(
    model,
    ax=ax,
    q_smooth=0.5,
    q_min=Q_MIN,
    q_max=Q_MAX,
    descriptor_labels=DESCRIPTOR_LABELS,
    plot_error=True,
    norm_factors=NORM_FACTORS,
    q_bins=Q_BINS_GRAD,
    absolute=False,
)
ax.set_title(r"Signed descriptor gradients along $q$")
TAI_Tools.save_fig_pdf_and_png(fig, "HG_gradient_along_q", output_path=OUTPUT_DIR)

unit_fig = vis.plot_gradient_unit_along_q(
    model,
    q_ranges=[Q_MIN, -10, 0, Q_MAX],
    q_spacing=[5, 2, 5],
    columns=[(0, 3), (3, 6), (6, 9)],
    q_smooth=0.1,
    descriptor_labels=DESCRIPTOR_LABELS,
    plot_error=True,
    norm_factors=NORM_FACTORS,
    q_bins=Q_BINS_GRAD,
    figsize=(14, 8),
)
unit_fig.suptitle("Normalized descriptor-gradient importance along q")
TAI_Tools.save_fig_pdf_and_png(unit_fig, "HG_gradient_unit_along_q", output_path=OUTPUT_DIR)
plt.show()


In [ ]:
results_gradient = vis.compute_gradient_stats_per_q(
           model, norm_factors=NORM_FACTORS, q_bins=Q_BINS_GRAD )


In [ ]:
vis.plot_settings.fontsize = 20

In [ ]:
unit_fig = vis.plot_gradient_unit_and_magnitude_along_q(
    model,
    q_ranges=[Q_MIN, -10, 0, Q_MAX],
    q_spacing=[5, 2, 5],
    columns=[(0, 3), (3, 6), (6, 9)],
    q_smooth=0.5,
    descriptor_labels=DESCRIPTOR_LABELS,
    plot_error=True,
    norm_factors=NORM_FACTORS,
    q_bins=Q_BINS_GRAD,
    figsize=(14, 10),
)

# unit_fig.suptitle("Descriptor-gradient magnitude and normalized importance along q")

TAI_Tools.save_fig_pdf_and_png(
    unit_fig,
    "HG_gradient_magnitude_and_unit_along_q",
    output_path=OUTPUT_DIR,
)

plt.show()

## 8. Global and transition-state gradient importance


In [ ]:
gradient_global = weighted_gradient_importance(
    model,
    vis,
    X_sample,
    weights=w_sample,
    descriptor_names_list=names,
)
gradient_ts = weighted_gradient_importance(
    model,
    vis,
    X_sample,
    weights=w_sample,
    descriptor_names_list=names,
    q_center=0.0,
    q_width=1.0,
)
display(gradient_global)
display(gradient_ts)


## 9. Ordinary and conditional permutation importance


In [ ]:
permutation = weighted_permutation_importance(
    model,
    X_sample,
    shots_sample,
    weights=w_sample,
    descriptor_names_list=names,
    n_repeats=3,
)
conditional = conditional_permutation_importance(
    model,
    X_sample,
    shots_sample,
    weights=w_sample,
    descriptor_names_list=names,
    n_bins=20,
    n_repeats=3,
)
display(permutation)
display(conditional)


## 10. HIPR importance


In [ ]:
hipr_trainset = aimmd.TrainSet(
    descriptors=X_sample,
    shot_results=shots_sample,
    weights=w_sample,
    n_states=2,
)
hipr = aimmd.analysis.HIPRanalysis(model, hipr_trainset, n_redraw=5)
hipr_losses, hipr_stds = hipr.do_hipr_plus(25)
hipr_table = pd.DataFrame({
    "descriptor": names,
    "hipr_delta_loss": hipr_losses[:-1] - hipr_losses[-1],
    "hipr_std": hipr_stds[:-1],
}).sort_values("hipr_delta_loss", ascending=False).reset_index(drop=True)
display(hipr_table)


## 11. Combined importance summary


In [ ]:
summary = gradient_global.merge(
    gradient_ts[["descriptor", "weighted_mean_abs_gradient", "weighted_rms_gradient"]],
    on="descriptor",
    suffixes=("_global", "_ts"),
)
summary = summary.merge(permutation, on="descriptor")
summary = summary.merge(conditional, on="descriptor")
summary = summary.merge(hipr_table, on="descriptor")
summary = summary.sort_values("hipr_delta_loss", ascending=False).reset_index(drop=True)
display(summary)

metrics = [
    # ("permutation_importance_mean", "Permutation Δ loss"),
    # ("conditional_importance_mean", "Conditional Δ loss"),
    ("hipr_delta_loss", r"HIPR $\Delta$ loss"),

    ("weighted_mean_abs_gradient_global", r"$\langle|\partial q/\partial \xi|\rangle$"),
    # ("weighted_mean_abs_gradient_ts", "TS-gated mean $|\partial q/\partial x|"),
]
fig, axes = plt.subplots(len(metrics), 1, figsize=(5, 5), sharex=True)
x = np.arange(len(summary))
for ax, (column, label) in zip(axes, metrics):
    ax.bar(x, summary[column], )
    ax.set_ylabel(label)
    ax.grid(axis="y", alpha=0.2)
axes[-1].set_xticks(x)
axes[-1].set_xticklabels(summary["descriptor"], rotation=90, ha="right")

# fig.suptitle("Host–guest committor descriptor importance")


labels = ["A", "B", "C"]

offset_x = -15  # pixels
offset_y = 5   # pixels
from matplotlib.transforms import ScaledTranslation

for i, ax in enumerate(axes):

    trans = ax.transAxes + ScaledTranslation(
        offset_x / fig.dpi,
        offset_y / fig.dpi,
        fig.dpi_scale_trans
    )

    ax.text(
        0, 1,
        labels[i],
        transform=trans,
        fontsize=15,
        fontweight="bold",
        va="bottom",
        ha="right",
    )


fig.tight_layout()
TAI_Tools.save_fig_pdf_and_png(fig, "HG_descriptor_importance_summary", output_path=OUTPUT_DIR)
plt.show()
